# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 09: The Weisfeiler-Lehman (1-WL) Test & GIN Expressiveness

### 1. Conceptual Intuition & Theoretical Background
How expressive are Graph Neural Networks? Can standard message passing GNNs (GCN, GraphSAGE) differentiate any two non-isomorphic graphs?

**The 1-WL (Weisfeiler-Lehman) Isomorphism Test:**
1. Assign an initial color/label $c_v^{(0)}$ to every node.
2. Iteratively update colors by hashing a node's current color with the sorted multiset of its neighbors' colors:
$$c_v^{(t+1)} = \text{HASH}\left( c_v^{(t)},\ \{\{ c_u^{(t)} : u \in \mathcal{N}(v) \}\} \right)$$
3. Count the histogram of colors. If histograms differ, the graphs are definitely non-isomorphic.

**Morris et al. / Xu et al. (ICLR 2019) Upper Bound Theorem:**
Standard message passing GNNs are **at most as expressive as the 1-WL test**. Standard GCN and GraphSAGE fail to match 1-WL because their aggregators (mean-pooling, max-pooling) are **non-injective multiset functions** (e.g., mean-pooling cannot distinguish between multiset $\{1, 1\}$ and $\{1\}$).

**Graph Isomorphism Network (GIN):**
Uses a provably injective sum-aggregator parameterization:
$$h_v^{(k)} = \text{MLP}^{(k)}\left( (1 + \epsilon^{(k)}) h_v^{(k-1)} + \sum_{u \in \mathcal{N}(v)} h_u^{(k-1)} \right)$$

---

### 2. Scratch Python Implementation: 1-WL Color Refinement

In [ ]:
import hashlib
from collections import Counter

def wl_color_refinement_step(adj_matrix, colors):
    """Executes a single 1-WL color refinement hashing step."""
    n_nodes = len(adj_matrix)
    new_colors = []
    
    for v in range(n_nodes):
        neighbors = [colors[u] for u in range(n_nodes) if adj_matrix[v, u] == 1]
        neighbors.sort() # Sorted multiset
        
        # Combine current color with sorted neighbor multiset
        signature = f"{colors[v]}_" + "_".join(neighbors)
        # Hash signature
        hashed_color = hashlib.md5(signature.encode('utf-8')).hexdigest()[:6]
        new_colors.append(hashed_color)
        
    return new_colors

# Compare Two Non-Isomorphic Graphs with Identical Degrees
# Graph A: Two separate disconnected triangles (6 nodes total)
# Graph B: One single 6-node cycle (6 nodes total, each node degree 2)
import numpy as np

# Adjacency Graph A (Two 3-cycles)
A_graph = np.zeros((6, 6), dtype=int)
A_graph[0, 1] = A_graph[1, 0] = 1; A_graph[1, 2] = A_graph[2, 1] = 1; A_graph[2, 0] = A_graph[0, 2] = 1
A_graph[3, 4] = A_graph[4, 3] = 1; A_graph[4, 5] = A_graph[5, 4] = 1; A_graph[5, 3] = A_graph[3, 5] = 1

# Adjacency Graph B (One 6-cycle)
B_graph = np.zeros((6, 6), dtype=int)
for i in range(6):
    B_graph[i, (i + 1) % 6] = B_graph[(i + 1) % 6, i] = 1

# Initialize uniform colors
colors_A = ["C0"] * 6
colors_B = ["C0"] * 6

# Execute 1-WL refinement
colors_A = wl_color_refinement_step(A_graph, colors_A)
colors_B = wl_color_refinement_step(B_graph, colors_B)

hist_A = Counter(colors_A)
hist_B = Counter(colors_B)

print("--- 1-WL Graph Isomorphism Test ---")
print(f"Graph A (Two 3-node Triangles) Final Colors: {colors_A[:3]}")
print(f"Graph B (One 6-node Hexagon) Final Colors:   {colors_B[:3]}")
print(f"Are Color Multisets Identical?              {hist_A == hist_B}")
print("1-WL test proves the two graphs are structurally distinct!")

### 3. Step-by-Step Code Walkthrough
1. **Multiset Aggregation:** Gathers and sorts neighbor colors.
2. **Deterministic Hashing:** Assigns a new unique color based on neighborhood configuration.
3. **Injective Proof:** Shows why sum-aggregation is necessary to separate distinct multiset structures.

---

### 4. Industry Applications & Trade-offs
- **Molecular Property Prediction (Cheminformatics):** Differentiating chemical isomers that share identical atomic formulas.
- **Trade-off:** 1-WL cannot count small graph sub-structures (such as triangles or 4-cycles) without higher-order $k$-WL generalizations.

---

### 5. Top Interview Questions & Answers
**Q1: Why does mean-pooling fail to achieve the maximum expressiveness of the 1-WL test?**
> **Answer:** Mean-pooling calculates averages, which cannot distinguish between a node with two neighbors of color $c$ and a node with three neighbors of color $c$ (since $\text{mean}(\{c, c\}) = \text{mean}(\{c, c, c\})$). Only sum-pooling preserves multiset cardinality.

**Q2: Can standard GNNs differentiate between strongly regular graphs that are non-isomorphic?**
> **Answer:** No. The 1-WL test (and therefore standard message passing GNNs) cannot differentiate non-isomorphic strongly regular graphs with identical parameters (such as the Shrikhande graph vs the 4x4 Rook's graph). Higher-order (3-WL) GNNs are required.